# Mamba vs. Transformer Porównanie zdolności klasyfikacji na zbiorze danych IMDb

## 1. Setup i biblioteki

Możemy to pominąć i wykonać komendę `uv sync` jeśli notebook uruchamiamy lokalnie

### 1.1 Instalacja bibliotek

#### 1.1.1 Konkretna wersja pytorch

In [ ]:
%pip install --force-reinstall torch==2.5.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try 'pacman -S
    python-xyz', where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Arch-packaged Python package,
    create a virtual environment using 'python -m venv path/to/venv'.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip.
    
    If you wish to install a non-Arch packaged Python application,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. Make sure you have python-pipx
    installed via pacman.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detailed specification.


#### 1.1.2 Pobranie skombilowanej mamby


In [ ]:
import sys
import torch

py_ver = f"cp{sys.version_info.major}{sys.version_info.minor}"
torch_ver = ".".join(torch.__version__.split(".")[:2])

print(f"Detected Python: {py_ver}")
print(f"Detected PyTorch: torch{torch_ver}")

base_conv = f"causal_conv1d-1.6.0+cu12torch{torch_ver}cxx11abiFALSE-{py_ver}-{py_ver}-linux_x86_64.whl"
base_mamba = f"mamba_ssm-2.3.0+cu12torch{torch_ver}cxx11abiFALSE-{py_ver}-{py_ver}-linux_x86_64.whl"

conv_url = f"https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.0/{base_conv}"
mamba_url = f"https://github.com/state-spaces/mamba/releases/download/v2.3.0/{base_mamba}"

%pip install packaging ninja --quiet

print("\nUnpacking Causal-Conv1d...")
%pip install {conv_url} --no-build-isolation

print("\nUnpacking Mamba-SSM...")
%pip install {mamba_url} --no-build-isolation


#### 1.1.3 Reszta bibiliotek

In [ ]:
%pip install transformers datasets accelerate evaluate scikit-learn

## 1.2 Monkeypatch mamby

In [ ]:
# Targeted diagnostic
import pkgutil
import mamba_ssm.ops.triton as triton_pkg

print("=== mamba_ssm.ops.triton submodules ===")
print([m.name for m in pkgutil.iter_modules(triton_pkg.__path__)])

try:
    from mamba_ssm.ops.triton.selective_state_update import selective_state_update
    print("\nselective_state_update: FOUND in triton.selective_state_update")
except Exception as e:
    print(f"\nselective_state_update: NOT FOUND — {e}")

import causal_conv1d
print("\n=== causal_conv1d top-level ===")
print([x for x in dir(causal_conv1d) if not x.startswith("_")])

try:
    from causal_conv1d import causal_conv1d_update
    print("\ncausal_conv1d_update: FOUND")
except Exception as e:
    print(f"\ncausal_conv1d_update: NOT FOUND — {e}")

## 1.3 Importy

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import evaluate
import datasets
from transformers import (
    AutoTokenizer, AutoConfig,
    MambaPreTrainedModel, MambaModel,
    TrainingArguments, Trainer,
    DataCollatorWithPadding,
)
from transformers.modeling_outputs import SequenceClassifierOutput

In [ ]:
def compute_metrics(eval_pred):
    load_accuracy = evaluate.load("accuracy")
    load_f1 = evaluate.load("f1")
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = load_accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = load_f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": accuracy, "f1": f1}

## 2. Wczytanie datasetu i preprocessing


In [3]:
print("Is CUDA available?", torch.cuda.is_available())

Is CUDA available? True


In [4]:
dataset = datasets.load_dataset('stanfordnlp/imdb')

print("Dataset loaded successfully:")
print(dataset)

Dataset loaded successfully:
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [5]:
print("\nExample from training set:")
print(dataset['train'][0])


Example from training set:
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudi

## 3. Podejście 1: Pretrenowany transformer

Użycie Trainer z biblioteki transformers do fine-tuningu modelu - uczenie całego modeleu bez mrożenia wag. 
Używamy modelu `distilbert-base-uncased` do klasyfikacji sentymentu.

In [ ]:
import time

from transformers import (
    TrainerCallback,
    DistilBertForSequenceClassification,
    DistilBertConfig,
)


def fineTuneBert(
    max_length=512, batch_size=32, lr=2e-5, num_epochs=5, weight_decay=0.01
):
    tokenizer = AutoTokenizer.from_pretrained(
        "distilbert/distilbert-base-uncased"
    )

    def tokenize_function(examples):
        return tokenizer(
            examples["text"], truncation=True, max_length=max_length
        ) 

    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    tokenized_dataset = tokenized_dataset.remove_columns(["text"])
    tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
    tokenized_dataset.set_format("torch")

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    class TimingCallback(TrainerCallback):
        def on_epoch_begin(self, args, state, control, **kwargs):
            self.epoch_start_time = time.time()

        def on_epoch_end(self, args, state, control, **kwargs):
            epoch_end_time = time.time()
            epoch_duration = epoch_end_time - self.epoch_start_time
            print(
                f"Epoch {state.epoch:.0f} completed in {epoch_duration:.2f} seconds"
            )

    config = DistilBertConfig(
        max_position_embeddings=max_length,
        num_labels=2,
        problem_type="single_label_classification",
    )
    model = DistilBertForSequenceClassification(config=config)

    training_args = TrainingArguments(
        output_dir="./results",
        eval_strategy="epoch",
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        weight_decay=weight_decay,
        logging_dir="./logs",
        logging_steps=1, 
        report_to="tensorboard", 
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["test"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[
            TimingCallback()
        ],  
    )

    results = trainer.evaluate()
    print("\nDistilBERT Initial Evaluation Results:")
    print(results)

    trainer.train()

    results = trainer.evaluate()
    print("\nDistilBERT Evaluation Results:")
    print(results)

### Długość sekwencji 128

In [7]:
fineTuneBert(128)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


KeyboardInterrupt: 

### Długość sekwencji 256

In [ ]:
fineTuneBert(256)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 5 completed in 138.61 seconds


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.009905,0.321185,5,0.910040,0.910033



DistilBERT Evaluation Results:
{'eval_loss': 0.32118549942970276, 'eval_accuracy': 0.91004, 'eval_f1': 0.9100332216870016}


### Długość sekwencji 512

In [ ]:
fineTuneBert(512,batch_size=16)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.696974,0,0.498720,0.333471



DistilBERT Initial Evaluation Results:
{'eval_loss': 0.6969738006591797, 'eval_accuracy': 0.49872, 'eval_f1': 0.33347132795760187}


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.043056,0.218847,0.916800,0.916665
2,0.029927,0.222714,0.929080,0.929071
3,0.002444,0.284495,0.931640,0.931631
4,0.003637,0.366731,0.927040,0.926992
5,0.000929,0.357567,0.931720,0.931719


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 5 completed in 312.27 seconds


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000929,0.357567,5,0.931720,0.931719



DistilBERT Evaluation Results:
{'eval_loss': 0.35756716132164, 'eval_accuracy': 0.93172, 'eval_f1': 0.9317193854744693}


### Długość sekwencji 1024

In [ ]:
fineTuneBert(1024,batch_size=16,lr=1e-5,num_epochs=10,weight_decay=0.01)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.695970,0,0.500000,0.333333



DistilBERT Initial Evaluation Results:
{'eval_loss': 0.6959699988365173, 'eval_accuracy': 0.5, 'eval_f1': 0.33333333333333326}


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.266240,0.373753,0.836280,0.834751
2,0.036761,0.358681,0.850920,0.849386
3,0.014580,0.319574,0.880600,0.880594
4,0.009429,0.395163,0.878880,0.878821
5,0.006657,0.469261,0.874920,0.874769
6,0.419148,0.489335,0.874960,0.874908


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 6 completed in 589.02 seconds


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 4. Podejście 2: Pretrenowana Mamba

Używamy modelu `state-spaces/mamba-130m`, który potem fine-tunujemy do zadania klasyfikacji. Biblioteka `transformers` nie oferuje gotowego modelu do tego zadania, więc napisaliśmy własny wrapper 

In [8]:
from transformers import AutoTokenizer
from transformers import GPTNeoXTokenizerFast

def prepare_mamba_dataset(dataset, max_length=1024):
    # 1. Initialize the Mamba tokenizer
    tokenizer = GPTNeoXTokenizerFast.from_pretrained("state-spaces/mamba-130m-hf")

    # 2. CRITICAL FOR MAMBA: Assign the EOS token as the padding token
    tokenizer.pad_token = tokenizer.eos_token

    # 3. Define the mapping tokenization function
    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=max_length,
            return_attention_mask=True,
            # We don't pad here; DataCollatorWithPadding will handle it dynamically per batch
        )

    # 4. Apply tokenization across the dataset splits
    print("Tokenizing dataset for Mamba...")
    tokenized_dataset = dataset.map(tokenize_function, batched=True)

    # 5. Format columns to match what the Trainer and PyTorch expect
    tokenized_dataset = tokenized_dataset.remove_columns(["text"])
    if "label" in tokenized_dataset["train"].column_names:
        tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

    tokenized_dataset.set_format("torch")

    return tokenized_dataset, tokenizer

In [9]:
# ── Patch: mamba-ssm v2.x removed top-level ops that transformers still expects ──
import mamba_ssm

if not hasattr(mamba_ssm, 'selective_state_update'):
    from mamba_ssm.ops.selective_scan_interface import (
        selective_scan_fn,
        selective_state_update,
    )
    mamba_ssm.selective_state_update = selective_state_update
    mamba_ssm.selective_scan_fn     = selective_scan_fn

if not hasattr(mamba_ssm, 'mamba_inner_fn'):
    try:
        from mamba_ssm.ops.selective_scan_interface import mamba_inner_fn
        mamba_ssm.mamba_inner_fn = mamba_inner_fn
    except ImportError:
        # v2 renamed / removed mamba_inner_fn; None makes transformers skip it
        mamba_ssm.mamba_inner_fn = None

print("mamba_ssm patched:",
      hasattr(mamba_ssm, 'selective_state_update'),
      hasattr(mamba_ssm, 'selective_scan_fn'),
      hasattr(mamba_ssm, 'mamba_inner_fn'))

mamba_ssm patched: True True True


In [10]:
class MambaForSequenceClassification(MambaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.backbone = MambaModel(config)  # must be 'backbone' to match checkpoint keys
        self.score = nn.Linear(config.hidden_size, self.num_labels, bias=False)
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.backbone(input_ids=input_ids, use_cache=False)
        hidden  = outputs[0]  # (B, L, H)

        if attention_mask is not None:
            seq_lens = attention_mask.int().sum(-1) - 1
            pooled   = hidden[torch.arange(hidden.size(0), device=hidden.device), seq_lens]
        else:
            pooled   = hidden[:, -1, :]

        logits = self.score(pooled)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits.view(-1, self.num_labels), labels.view(-1))

        return SequenceClassifierOutput(loss=loss, logits=logits, hidden_states=outputs.hidden_states)

In [11]:
import time

from transformers import AutoTokenizer, TrainerCallback, TrainingArguments, Trainer, AutoConfig

class TimingCallback(TrainerCallback):
        def on_epoch_begin(self, args, state, control, **kwargs):
            self.epoch_start_time = time.time()

        def on_epoch_end(self, args, state, control, **kwargs):
            epoch_end_time = time.time()
            epoch_duration = epoch_end_time - self.epoch_start_time
            print(f"Epoch {state.epoch:.0f} completed in {epoch_duration:.2f} seconds")

def fineTuneMambaClassification(tokenized_dataset, collator,batch_size=16):
    # 1. Initialize Tokenizer & handle padding token requirements
    tokenizer = AutoTokenizer.from_pretrained("state-spaces/mamba-130m-hf")
    tokenizer.pad_token = tokenizer.eos_token

    # 2. Initialize our custom classification model
    # Load the configuration with use_mamba_kernels set to False from the start
    config = AutoConfig.from_pretrained(
        "state-spaces/mamba-130m-hf",
        num_labels=2,
        use_mamba_kernels=True # Set this directly during config loading
    )

    model = MambaForSequenceClassification.from_pretrained(
        "state-spaces/mamba-130m-hf",
        config=config, # Pass the fully configured config
        ignore_mismatched_sizes=True
    )
    # Explicitly configure pad token ID to match the tokenizer configuration
    model.config.pad_token_id = tokenizer.pad_token_id


    # 3. Define Standard Training Arguments
    training_args = TrainingArguments(
        output_dir="./mamba_classification_results",
        eval_strategy="epoch",
        learning_rate=3e-5,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=5,
        gradient_checkpointing=False,
        weight_decay=0.01,
        logging_steps=1,
        fp16=False,  # Recommended for custom CUDA compilation layouts
        bf16=True,  # Recommended for custom CUDA compilation layouts
        report_to="tensorboard"
    )

    # 4. Fire up the regular Hugging Face Trainer engine
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["test"],
        data_collator=collator,
        compute_metrics=compute_metrics, # Pass your accuracy/f1 metric helper
        callbacks=[TimingCallback()] # Add TensorBoardCallback and TimingCallback
    )

    trainer.train()

### Długość sekwencji 128

In [ ]:
import sys
import torch # Import torch to get its __file__ attribute
from transformers import DataCollatorWithPadding


sys.modules['__main__'].__file__ = torch.__file__ # Point to an actual file from an imported module

# 1. Process data for Mamba (supporting 1024 text length safely!)
tokenized_dataset, mamba_tokenizer = prepare_mamba_dataset(dataset, max_length=128)

# 2. Build the collator with the explicit Mamba tokenizer reference
data_collator = DataCollatorWithPadding(tokenizer=mamba_tokenizer)

# 3. Pass tokenized_dataset and data_collator into your trainer!
fineTuneMambaClassification(tokenized_dataset,data_collator,batch_size=16)

Tokenizing dataset for Mamba...


Loading weights: 100%|██████████| 242/242 [00:00<00:00, 22744.56it/s]
[transformers] MambaForSequenceClassification LOAD REPORT from: state-spaces/mamba-130m-hf
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.092220,0.266649,0.894240,0.894170
2,0.002116,0.375544,0.893840,0.893787
3,0.000026,0.664315,0.893880,0.893869


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.17it/s]


Epoch 1 completed in 164.73 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.53it/s]


Epoch 2 completed in 164.54 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.43it/s]


Epoch 3 completed in 164.82 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.68it/s]


### Długość sekwencji 256

In [25]:
import sys
import torch # Import torch to get its __file__ attribute
from transformers import DataCollatorWithPadding


sys.modules['__main__'].__file__ = torch.__file__ # Point to an actual file from an imported module

# 1. Process data for Mamba (supporting 1024 text length safely!)
tokenized_dataset, mamba_tokenizer = prepare_mamba_dataset(dataset, max_length=256)

# 2. Build the collator with the explicit Mamba tokenizer reference
data_collator = DataCollatorWithPadding(tokenizer=mamba_tokenizer)

# 3. Pass tokenized_dataset and data_collator into your trainer!
fineTuneMambaClassification(tokenized_dataset,data_collator,batch_size=16)

Tokenizing dataset for Mamba...


Loading weights: 100%|██████████| 242/242 [00:00<00:00, 14609.67it/s]
[transformers] MambaForSequenceClassification LOAD REPORT from: state-spaces/mamba-130m-hf
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.123468,0.201960,0.922920,0.922892
2,0.005053,0.254438,0.924240,0.924193
3,0.000107,0.395591,0.925760,0.925757
4,0.000044,0.534691,0.927400,0.927397
5,0.000035,0.573302,0.927600,0.927599


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.69it/s]


Epoch 1 completed in 253.29 seconds


...

Epoch 3 completed in 253.70 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.31it/s]


Epoch 4 completed in 253.79 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.50it/s]


Epoch 5 completed in 255.26 seconds


### Długość sekwencji 512

In [26]:
import sys
import torch 
from transformers import DataCollatorWithPadding


sys.modules['__main__'].__file__ = torch.__file__ 

tokenized_dataset, mamba_tokenizer = prepare_mamba_dataset(dataset, max_length=512)

data_collator = DataCollatorWithPadding(tokenizer=mamba_tokenizer)

fineTuneMambaClassification(tokenized_dataset,data_collator,batch_size=16)

Tokenizing dataset for Mamba...


Loading weights: 100%|██████████| 242/242 [00:00<00:00, 17198.80it/s]
[transformers] MambaForSequenceClassification LOAD REPORT from: state-spaces/mamba-130m-hf
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.029462,0.168756,0.940000,0.939974
2,0.019181,0.182540,0.944120,0.944112
3,0.000020,0.325445,0.942840,0.942834
4,0.000719,0.412771,0.945440,0.945440
5,0.000001,0.441448,0.944880,0.944879


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.54it/s]


Epoch 1 completed in 445.90 seconds


...

Epoch 3 completed in 450.96 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.35it/s]


Epoch 4 completed in 444.77 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.44it/s]


Epoch 5 completed in 446.99 seconds


### Długość sekwencji 1024

In [12]:
import sys
import torch 
from transformers import DataCollatorWithPadding


sys.modules['__main__'].__file__ = torch.__file__ 

tokenized_dataset, mamba_tokenizer = prepare_mamba_dataset(dataset, max_length=1024)

data_collator = DataCollatorWithPadding(tokenizer=mamba_tokenizer)

fineTuneMambaClassification(tokenized_dataset,data_collator,batch_size=8)

Tokenizing dataset for Mamba...


Loading weights: 100%|██████████| 242/242 [00:00<00:00, 10802.12it/s]
[transformers] MambaForSequenceClassification LOAD REPORT from: state-spaces/mamba-130m-hf
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.158752,0.201754,0.940840,0.940798
2,0.002095,0.217714,0.948840,0.948835
3,0.000056,0.343092,0.947160,0.947154
4,0.004730,0.571780,0.945680,0.945679
5,0.000000,0.608669,0.947520,0.947520


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.35it/s]


Epoch 1 completed in 713.34 seconds


...

Epoch 3 completed in 760.00 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.09it/s]


Epoch 4 completed in 760.32 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.26it/s]


Epoch 5 completed in 728.29 seconds


## 5. Analiza skalowalności

This section will focus on comparing both architectures across different sequence lengths (128, 512, 1024 tokens). We will measure:
-   **Training time per epoch**
-   **Inference time**
-   **Quality metrics**: Accuracy, F1-score

This will involve re-tokenizing the dataset with different `max_length` values and repeating the training and evaluation steps for each model and sequence length.